# 04. Sequential Deep Learning: 1D CNN + LSTM Architecture
**Objective**: Implement and evaluate a hybrid deep learning model combining 1D Convolutions with stacked LSTM units for temporal feature extraction and sequence modeling.

In [ ]:
import os
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Compute Device: {DEVICE}")

class CNN_LSTM(nn.Module):
    def __init__(self, input_dim):
        super(CNN_LSTM, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        self.lstm = nn.LSTM(input_size=64, hidden_size=50, num_layers=2, batch_first=True, dropout=0.2)
        self.fc1 = nn.Linear(50, 25)
        self.fc2 = nn.Linear(25, 1)
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.pool(self.relu(self.conv1(x)))
        x = x.permute(0, 2, 1)
        out, (hn, cn) = self.lstm(x)
        x = self.relu(out[:, -1, :])
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x.squeeze()

model = CNN_LSTM(input_dim=94).to(DEVICE)
print("Model Architecture Summary:")
print(model)

## 1. Load Trained Checkpoint Weights[cite: 1, 2]

In [ ]:
MODEL_PATH = "../models/cnn_lstm_model.pt"
if os.path.exists(MODEL_PATH):
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    print(f"Model checkpoint successfully loaded from {MODEL_PATH}!")
else:
    print(f"Checkpoint not found at {MODEL_PATH}.")